# 🍽️ Nutrition5k CNN Training
**Train a MobileNetV2 nutrition regressor on the full Nutrition5k dataset**

This notebook:
1. Downloads ~5,006 overhead food images directly from Google Cloud Storage (~2-3 GB)
2. Prepares the dataset (CSV + image conversion)
3. Trains a MobileNetV2 CNN for 60 epochs
4. Exports `nutrition_cnn.pkl` for the Smart Medical System backend

⚡ **Runtime**: Make sure you select **GPU** → Runtime > Change runtime type > **T4 GPU**

⏱️ **Total time**: ~30-60 minutes

## Step 1: Download Dataset from Google Cloud Storage
Downloads ONLY overhead images + metadata (~2-3 GB), NOT the full 180GB dataset.

In [ ]:
import os
import subprocess

BASE = "gs://nutrition5k_dataset/nutrition5k_dataset"
LOCAL = "/content/nutrition5k"

os.makedirs(f"{LOCAL}/images", exist_ok=True)

# 1a. Download metadata CSVs
print("[1/3] Downloading metadata CSVs...")
!gsutil -m cp -r {BASE}/metadata/ {LOCAL}/metadata/

# 1b. Download train/test splits
print("\n[2/3] Downloading train/test split files...")
!gsutil -m cp -r {BASE}/dish_ids/ {LOCAL}/dish_ids/

# 1c. Download overhead RGB images (~2-3 GB)
print("\n[3/3] Downloading overhead RGB images (5-10 min)...")
!gsutil -m cp -r {BASE}/imagery/realsense_overhead/ {LOCAL}/imagery/realsense_overhead/

print("\n✅ Download complete!")

## Step 2: Prepare Dataset
Convert the Nutrition5k structure into the format our CNN expects:
- Combine metadata CSVs into `dishes.csv`
- Convert overhead PNG images to flat JPEG folder
- Copy train/test split files

In [ ]:
import pandas as pd
from PIL import Image
import shutil

LOCAL = "/content/nutrition5k"

# ── 2a. Build dishes.csv from Nutrition5k metadata ──
print("Building dishes.csv...")
dfs = []
for csv_file in ["dish_metadata_cafe1.csv", "dish_metadata_cafe2.csv"]:
    path = f"{LOCAL}/metadata/{csv_file}"
    if os.path.exists(path):
        df = pd.read_csv(path, header=None)
        df = df.iloc[:, :6]
        df.columns = ["dish_id", "total_calories", "total_mass", "total_fat", "total_carb", "total_protein"]
        dfs.append(df)
        print(f"  Loaded {len(df)} dishes from {csv_file}")

combined = pd.concat(dfs, ignore_index=True)
combined = combined.dropna(subset=["total_calories", "total_fat", "total_carb", "total_protein"])
combined = combined[combined["total_calories"] > 0]
combined.to_csv(f"{LOCAL}/dishes.csv", index=False)
print(f"\ndishes.csv: {len(combined)} dishes")

# ── 2b. Convert overhead PNGs to flat JPEG folder ──
print("\nConverting overhead images to JPEG...")
src_dir = f"{LOCAL}/imagery/realsense_overhead"
dst_dir = f"{LOCAL}/images"
count = 0
errors = 0

if os.path.exists(src_dir):
    for dish_dir in sorted(os.listdir(src_dir)):
        dish_path = os.path.join(src_dir, dish_dir)
        if not os.path.isdir(dish_path):
            continue
        # Find RGB image
        rgb_path = None
        for name in ["rgb.png", "rgb.jpg", "overhead_color.png", "color.png"]:
            p = os.path.join(dish_path, name)
            if os.path.exists(p):
                rgb_path = p
                break
        if not rgb_path:
            errors += 1
            continue
        dst_path = os.path.join(dst_dir, f"{dish_dir}.jpg")
        if os.path.exists(dst_path):
            count += 1
            continue
        try:
            img = Image.open(rgb_path).convert("RGB")
            img.save(dst_path, "JPEG", quality=95)
            count += 1
        except:
            errors += 1

print(f"Converted {count} images ({errors} errors)")

# ── 2c. Copy split files ──
print("\nCopying split files...")
splits_dir = f"{LOCAL}/dish_ids/splits"
if os.path.exists(splits_dir):
    for f in os.listdir(splits_dir):
        if "train" in f.lower() or "test" in f.lower():
            shutil.copy2(os.path.join(splits_dir, f), f"{LOCAL}/{f}")
            print(f"  Copied {f}")

# Verify
img_count = len([f for f in os.listdir(dst_dir) if f.endswith('.jpg')])
print(f"\n✅ Dataset ready: {img_count} images")

## Step 3: Define Model & Dataset Classes

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import transforms, models
import numpy as np
import pickle
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")


class NutritionCNN(nn.Module):
    def __init__(self, num_outputs=4, freeze_layers=80):
        super().__init__()
        self.backbone = models.mobilenet_v2(weights='IMAGENET1K_V1')

        for i, (name, param) in enumerate(self.backbone.named_parameters()):
            if i < freeze_layers:
                param.requires_grad = False

        in_features = self.backbone.classifier[1].in_features  # 1280
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(p=0.3),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(p=0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_outputs),
        )

    def forward(self, x):
        return self.backbone(x)

    def unfreeze_all(self):
        for param in self.backbone.parameters():
            param.requires_grad = True
        total = sum(p.numel() for p in self.parameters())
        print(f"Unfroze all layers ({total:,} params)")


class NutritionDataset(Dataset):
    def __init__(self, csv_path, img_dir, dish_ids_file, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

        with open(dish_ids_file) as f:
            valid_ids = set(line.strip() for line in f)

        existing_imgs = set(
            f.replace('.jpg', '') for f in os.listdir(img_dir) if f.endswith('.jpg')
        )
        usable_ids = valid_ids.intersection(existing_imgs)
        self.df = self.df[self.df['dish_id'].isin(usable_ids)].reset_index(drop=True)

        self.targets = ['total_calories', 'total_fat', 'total_carb', 'total_protein']
        self.means = self.df[self.targets].mean()
        self.stds = self.df[self.targets].std()

        print(f"Dataset: {len(self.df)} dishes")
        print(f"Means: cal={self.means['total_calories']:.1f}, fat={self.means['total_fat']:.1f}, carb={self.means['total_carb']:.1f}, prot={self.means['total_protein']:.1f}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{row['dish_id']}.jpg")
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB', (224, 224))
        if self.transform:
            img = self.transform(img)
        raw = row[self.targets].values.astype(np.float32)
        means = self.means.values.astype(np.float32)
        stds = self.stds.values.astype(np.float32)
        label = (raw - means) / (stds + 1e-8)
        return img, torch.tensor(label, dtype=torch.float32)


print("✅ Model & Dataset classes defined")

## Step 4: Train the Model
- Phase 1 (epochs 1-9): Frozen backbone, train only the classifier head
- Phase 2 (epochs 10+): Unfreeze all layers, fine-tune with lower LR
- Early stopping with patience=12

In [ ]:
# ── Config ──
EPOCHS = 60
BATCH_SIZE = 32
LR = 1e-3
LR_FINETUNE = 1e-5
UNFREEZE_EPOCH = 10
IMG_SIZE = 224
PATIENCE = 12
USE_AMP = True

LOCAL = "/content/nutrition5k"

# ── Transforms ──
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── Dataset ──
dataset = NutritionDataset(
    csv_path=f"{LOCAL}/dishes.csv",
    img_dir=f"{LOCAL}/images",
    dish_ids_file=f"{LOCAL}/train_ids.txt",
    transform=train_transform
)

val_size = int(0.15 * len(dataset))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
val_ds.dataset.transform = val_transform

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {train_size} | Val: {val_size}")

# ── Model ──
model = NutritionCNN(num_outputs=4, freeze_layers=80).to(DEVICE)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
scaler = torch.amp.GradScaler('cuda') if USE_AMP else None

def criterion(preds, targets):
    return nn.MSELoss()(preds, targets) + 0.5 * nn.L1Loss()(preds, targets)

# ── Training Loop ──
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'lr': []}
patience_count = 0
finetuning = False

print(f"\nStarting training for {EPOCHS} epochs...")
print("=" * 60)

for epoch in range(EPOCHS):
    if epoch + 1 == UNFREEZE_EPOCH and not finetuning:
        model.unfreeze_all()
        for g in optimizer.param_groups:
            g['lr'] = LR_FINETUNE
        finetuning = True

    # Train
    model.train()
    train_loss = 0.0
    for imgs, labels in tqdm(train_loader, desc=f"Ep {epoch+1}/{EPOCHS} [Train]", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            preds = model(imgs)
            loss = criterion(preds, labels)
        if scaler:
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        train_loss += loss.item()

    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f"Ep {epoch+1}/{EPOCHS} [Val]", leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                preds = model(imgs)
                val_loss += criterion(preds, labels).item()

    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(lr)
    scheduler.step(val_loss)

    tag = 'FT' if finetuning else 'FR'
    print(f"[{tag}] Ep {epoch+1:02d}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {lr:.6f}", end='')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_count = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
            'history': history,
        }, 'best_checkpoint.pth')
        print(f" ★ BEST")
    else:
        patience_count += 1
        print(f" ({patience_count}/{PATIENCE})")

    if patience_count >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print("\n✅ Training complete!")

## Step 5: Export Model & Download
Exports `nutrition_cnn.pkl` with the model weights + normalization stats.

In [ ]:
# Load best checkpoint
best_ckpt = torch.load('best_checkpoint.pth', map_location=DEVICE)
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()

# Export
pkl_payload = {
    'model_state_dict': model.state_dict(),
    'means': dataset.means.to_dict(),
    'stds': dataset.stds.to_dict(),
    'targets': dataset.targets,
    'img_size': IMG_SIZE,
    'history': best_ckpt['history'],
    'best_val_loss': best_ckpt['best_val_loss']
}

with open('nutrition_cnn.pkl', 'wb') as f:
    pickle.dump(pkl_payload, f)

print(f"✅ Saved: nutrition_cnn.pkl")
print(f"   Best val loss: {best_ckpt['best_val_loss']:.4f}")
print(f"   Epochs trained: {len(best_ckpt['history']['train_loss'])}")
print(f"   Targets: {dataset.targets}")
print(f"   Means: {dict(dataset.means)}")
print(f"   Stds: {dict(dataset.stds)}")

# Auto-download in Colab
try:
    from google.colab import files
    files.download('nutrition_cnn.pkl')
    print("\n📥 Download started! Place the file in:")
    print("   System_medical_assistant/models/nutrition_cnn.pkl")
except:
    print("\n📥 Download nutrition_cnn.pkl manually and place in:")
    print("   System_medical_assistant/models/nutrition_cnn.pkl")

## Step 6: Plot Training History (Optional)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['lr'], label='Learning Rate', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('LR')
ax2.set_title('Learning Rate Schedule')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()
print(f"Best val loss: {min(history['val_loss']):.4f} at epoch {history['val_loss'].index(min(history['val_loss']))+1}")